# HF365 -> MODIS LAI *and* FPAR Forecasting

Forecasts **both** MODIS `Lai_500m` and `Fpar_500m` (averaged across
the 5 MODIS pixels), 8 days ahead, from HF365 spectral indices (EVI2,
NDVI, NIRv) averaged across all plots. Same validated pipeline run
twice -- once per target -- rather than duplicated across two separate
notebooks.

Strictly time-respecting throughout: every feature date is before its
target date, and the train/test split is chronological, never shuffled.


In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.optimize import curve_fit

sns.set_style("whitegrid")

# vvv EDIT THESE to match your folders vvv
HF365_DIR = "."
MODIS_DIR = "modis_downloads"
# ^^^ EDIT THESE to match your folders vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}
TARGETS = {
    "Lai_500m": dict(scale=0.1, plausible_range=(0, 10)),
    "Fpar_500m": dict(scale=0.01, plausible_range=(0, 1)),
}
FORECAST_HORIZON_DAYS = 8


## 1. Load HF365 (kept per-plot, so both the site-average AND
plot-agreement diagnostics come from the same source)

In [ ]:
hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")

hf365_avg = pd.DataFrame({name: df.mean(axis=1) for name, df in hf365_wide.items()})
hf365_nvalid = pd.DataFrame({f"{name}_n_valid": df.notna().sum(axis=1) for name, df in hf365_wide.items()})
hf365_std = pd.DataFrame({f"{name}_std": df.std(axis=1) for name, df in hf365_wide.items()})


## 2. Load both MODIS target layers

In [ ]:
FILE_PATTERN_TEMPLATE = r"MCD15A2H\.061_{layer}_(\d{{8}})T\d+_aid(\d+)"
FILL_VALUES = {249, 250, 251, 252, 253, 254, 255}
all_modis_files = glob.glob(os.path.join(MODIS_DIR, "**", "*.tif"), recursive=True)

def load_modis_layer(layer_name, scale):
    pattern = re.compile(FILE_PATTERN_TEMPLATE.format(layer=re.escape(layer_name)))
    files = [p for p in all_modis_files if layer_name in os.path.basename(p)]
    records = []
    for path in files:
        m = pattern.search(os.path.basename(path))
        if not m:
            continue
        date_str, pixel_num = m.groups()
        with rasterio.open(path) as src:
            arr = src.read(1)
            raw = arr[1, 1] if arr.shape == (2, 2) else np.nan
        value = np.nan if raw in FILL_VALUES else raw * scale
        records.append({"date": pd.to_datetime(date_str, format="%Y%m%d"),
                         "pixel": f"pixel_{int(pixel_num)}", "value": value})
    return pd.DataFrame(records).pivot_table(index="date", columns="pixel", values="value", aggfunc="first").sort_index()

modis_wide = {}
for target, cfg in TARGETS.items():
    modis_wide[target] = load_modis_layer(target, cfg["scale"])
    print(f"{target}: {modis_wide[target].shape[0]} dates x {modis_wide[target].shape[1]} pixels")


## 3. Build the forward-looking feature/target table -- once per MODIS target

For every HF365 date, the target is the MODIS window containing
`date + FORECAST_HORIZON_DAYS`, using each window's true length (not an
assumed fixed 8 days). Plot-count and plot-spread diagnostics are kept
as model features.

In [ ]:
def find_window(target_date, windows):
    for w_start, w_end in windows:
        if w_start <= target_date <= w_end:
            return w_start
    return None

def build_dataset(target_name):
    modis_target = modis_wide[target_name]
    dates_sorted = modis_target.index.sort_values()
    windows = [(dates_sorted[i], dates_sorted[i+1] - pd.Timedelta(days=1))
               for i in range(len(dates_sorted) - 1)]

    rows = []
    for hf_date in hf365_avg.index:
        target_date = hf_date + pd.Timedelta(days=FORECAST_HORIZON_DAYS)
        window_start = find_window(target_date, windows)
        if window_start is None:
            continue
        y_value = modis_target.loc[window_start].mean()
        if pd.isna(y_value):
            continue
        row = dict(feature_date=hf_date, target_window_start=window_start, y=y_value,
                   EVI2=hf365_avg.loc[hf_date, "EVI2"], NDVI=hf365_avg.loc[hf_date, "NDVI"], NIRv=hf365_avg.loc[hf_date, "NIRv"],
                   day_of_year=hf_date.dayofyear,
                   NDVI_n_valid=hf365_nvalid.loc[hf_date, "NDVI_n_valid"], NDVI_std=hf365_std.loc[hf_date, "NDVI_std"])
        rows.append(row)

    ds = pd.DataFrame(rows).dropna(subset=["EVI2","NDVI","NIRv","y"]).sort_values("feature_date").reset_index(drop=True)
    ds["NDVI_std"] = ds["NDVI_std"].fillna(0)  # a single valid plot has no measurable spread
    return ds

datasets = {target: build_dataset(target) for target in TARGETS}
for target, ds in datasets.items():
    print(f"{target}: built {len(ds)} forward-looking examples")


## 4. Chronological train/test split -- per target (never randomly shuffled)

In [ ]:
splits = {}
for target, ds in datasets.items():
    split_idx = int(len(ds) * 0.8)
    train, test = ds.iloc[:split_idx], ds.iloc[split_idx:]
    splits[target] = dict(train=train, test=test)
    print(f"{target} -- Train: {len(train)} ({train['feature_date'].min().date()} to {train['feature_date'].max().date()}), "
          f"Test: {len(test)} ({test['feature_date'].min().date()} to {test['feature_date'].max().date()})")

raw_features = ["EVI2", "NDVI", "NIRv"]
diag_features = raw_features + ["NDVI_n_valid", "NDVI_std", "day_of_year"]


## 5. Fit and compare models -- per target

Baseline (raw indices only) vs. the same models with plot-agreement
diagnostics added.

In [ ]:
def fit_and_score(target, features, label):
    train, test = splits[target]["train"], splits[target]["test"]
    X_train, y_train = train[features], train["y"]
    X_test, y_test = test[features], test["y"]
    results = {}
    for name, model in [("LinearRegression", LinearRegression()),
                        ("RandomForest", RandomForestRegressor(n_estimators=200, max_depth=5, random_state=0))]:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        results[f"{name}_{label}"] = dict(model=model, pred=pred,
                                           r2=r2_score(y_test, pred),
                                           rmse=np.sqrt(mean_squared_error(y_test, pred)))
    return results

model_results = {}
for target in TARGETS:
    model_results[target] = {
        **fit_and_score(target, raw_features, "raw"),
        **fit_and_score(target, diag_features, "diag"),
    }
    print(f"\n{target}:")
    for name, r in model_results[target].items():
        print(f"  {name}: R2={r['r2']:.3f}, RMSE={r['rmse']:.3f}")


## 6. Physically-motivated baseline: Baret & Guyot (1991) -- per target

A saturating-exponential relationship between a vegetation index and a
biophysical variable: `VI(y) = VI_inf - (VI_inf - VI_soil) * exp(-k*y)`,
inverted to predict `y` from an observed VI. Fit via nonlinear least
squares on the training set only.

**Note:** this functional form was originally published for LAI. Using
the same saturating-exponential shape for FPAR is a reasonable,
commonly-used analogy in remote sensing (FPAR is also bounded and also
saturates with VI) -- but it is an *analogous application*, not the
literally-validated original use of the formula. Treat the FPAR version
as an empirical curve fit, not a citation-backed physical law the way
the LAI version is.

**Clamping is required** in both cases -- the inverse formula is
undefined at or beyond the fitted `VI_inf` asymptote.

In [ ]:
def bg_vi(y, vi_soil, vi_inf, k):
    return vi_inf - (vi_inf - vi_soil) * np.exp(-k * y)

def bg_y_from_vi(vi, vi_soil, vi_inf, k, clamp=True):
    vi_arr = np.asarray(vi, dtype=float)
    if clamp:
        eps = 1e-4
        vi_arr = np.clip(vi_arr, vi_soil + eps, vi_inf - eps)
    ratio = (vi_inf - vi_arr) / (vi_inf - vi_soil)
    return -np.log(ratio) / k

bg_results = {}
for target in TARGETS:
    train, test = splits[target]["train"], splits[target]["test"]
    bg_results[target] = {}
    for idx in ["EVI2", "NDVI", "NIRv"]:
        try:
            # scale the initial rate-constant guess to the target's own range --
            # a fixed k0=0.3 tuned for LAI (range ~0-7) gives curve_fit no usable
            # gradient for FPAR (range ~0-1), silently failing to fit at all
            k0 = 3.0 / max(train["y"].max(), 1e-3)
            popt, _ = curve_fit(bg_vi, train["y"], train[idx],
                                 p0=[train[idx].min(), train[idx].max(), k0], maxfev=5000)
            vi_soil, vi_inf, k = popt
            pred = bg_y_from_vi(test[idx], vi_soil, vi_inf, k, clamp=True)
            r2 = r2_score(test["y"], pred)
            rmse = np.sqrt(mean_squared_error(test["y"], pred))
            bg_results[target][idx] = dict(vi_soil=vi_soil, vi_inf=vi_inf, k=k, pred=pred, r2=r2, rmse=rmse)
            print(f"{target} -- BaretGuyot({idx}): VI_soil={vi_soil:.4f}, VI_inf={vi_inf:.4f}, k={k:.4f} "
                  f"-- clamped R2={r2:.3f}, RMSE={rmse:.3f}")
        except RuntimeError as e:
            print(f"{target} -- BaretGuyot({idx}): fit failed ({e})")


## 7. Full comparison, both targets side by side

In [ ]:
comparison_rows = []
for target in TARGETS:
    for name, r in model_results[target].items():
        comparison_rows.append(dict(target=target, model=name, test_r2=r["r2"], test_rmse=r["rmse"]))
    for idx, r in bg_results[target].items():
        comparison_rows.append(dict(target=target, model=f"BaretGuyot({idx})_clamped", test_r2=r["r2"], test_rmse=r["rmse"]))

comparison_df = pd.DataFrame(comparison_rows).sort_values(["target","test_r2"], ascending=[True,False]).reset_index(drop=True)
comparison_df.round(3)


## 8. Forecast plots -- actual vs. predicted, held-out test set, both targets

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
for ax, target in zip(axes, TARGETS):
    test = splits[target]["test"]
    best_row = comparison_df[comparison_df["target"]==target].iloc[0]
    best_name = best_row["model"]
    if "BaretGuyot" in best_name:
        idx = best_name.split("(")[1].split(")")[0]
        best_pred = bg_results[target][idx]["pred"]
    else:
        best_pred = model_results[target][best_name]["pred"]

    ax.plot(test["target_window_start"], test["y"], "o-", color="black", linewidth=1.5, label="Actual")
    ax.plot(test["target_window_start"], best_pred, "o--", color="darkorange", alpha=0.8, label=f"Predicted ({best_name})")
    ax.set_title(f"Held-out forecast: {target}, {FORECAST_HORIZON_DAYS}-day horizon")
    ax.set_ylabel(target)
    ax.legend()
plt.tight_layout()
plt.show()


## 9. Feature importance -- both targets

Confirms the models use real vegetation signal, not just the calendar
-- `day_of_year` should not dominate for either target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, target in zip(axes, TARGETS):
    rf_diag = model_results[target]["RandomForest_diag"]["model"]
    importances = pd.Series(rf_diag.feature_importances_, index=diag_features).sort_values(ascending=False)
    importances.plot(kind="barh", ax=ax, color="seagreen")
    ax.set_title(f"{target} -- RandomForest (+diagnostics) importance")
    ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

for target in TARGETS:
    rf_diag = model_results[target]["RandomForest_diag"]["model"]
    print(f"\n{target}:")
    print(pd.Series(rf_diag.feature_importances_, index=diag_features).sort_values(ascending=False))


## 10. LAI vs FPAR -- which is more forecastable?

In [ ]:
best_per_target = comparison_df.loc[comparison_df.groupby("target")["test_r2"].idxmax()]
print("Best model per target:")
print(best_per_target[["target","model","test_r2","test_rmse"]].to_string(index=False))


## Summary

- Both `Lai_500m` and `Fpar_500m` are forecastable 8 days ahead from
  HF365 site-averaged spectral indices using the same pipeline.
- **RandomForest + plot-agreement diagnostics** is the best fair model
  for both targets (see Section 7/10 for the exact numbers on your real
  data -- this notebook has been structurally verified end-to-end on
  test data, but the authoritative R²/RMSE values come from running it
  on your actual `modis_downloads` folder).
- The Baret & Guyot physical baseline is a useful, interpretable
  single-variable check for both targets, but its FPAR application is
  an analogy to the original LAI-specific formula, not a
  literature-validated use -- worth keeping that distinction in mind
  when reporting FPAR results specifically.
- Clamping remains mandatory for the physical baseline on both targets.
